# 04 — Classificação de 29/10/2020 e comparação com o artigo
Usa o modelo treinado no notebook 03 para classificar os endereços do dia de teste e compara com o artigo: **63 sinalizados, 3 Tornado (as três instâncias) e 60 falsos positivos**.

| Etapa | Saída |
|---|---|
| Classificação | probabilidade de mixer para cada endereço |
| Comparação com o artigo | `results/comparacao_artigo_20201029.csv` |
| Lista para análise dos falsos positivos | `results/sinalizados_20201029.csv` |
| Sensibilidade (limiar × MIN_TX) | `results/sensibilidade_limiar_mintx.csv` |
| Variação entre sementes | `results/variacao_sementes.csv` |

## Configuração

In [33]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd, numpy as np, json, time, requests, gc, warnings
warnings.filterwarnings("ignore", category=FutureWarning)

BASE = Path('/content/drive/MyDrive/mixers-ethereum')
RAW, INTERIM, PROC, RES = [BASE/p for p in ['data/raw','data/interim','data/processed','results']]
for p in [RAW, INTERIM, PROC, RES, BASE/'docs']:
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
TORNADO = {
    "0x12d66f87a04a9e220743712ce6d9bb1b5616b8fc": "0.1 ETH",
    "0x47ce0c6ed5b0ce3d3a51fdb1c52dc66a7c3c2936": "1 ETH",
    "0x910cbd523d972eb0a6f4cae4618ad62622b39dbf": "10 ETH",
}
T10 = "0x910cbd523d972eb0a6f4cae4618ad62622b39dbf"
FEATURES = ["value_entropy", "fee_entropy", "internal_value_entropy", "call_count_mean"]

# registro de decisões (mesma função do notebook 01)
dec = BASE/"docs/decisoes.md"
if not dec.exists():
    dec.write_text("# Decisões de reprodução\n\n| # | Tema | Decisão | Justificativa | Data |\n|---|---|---|---|---|\n")
def registrar(linha):
    if linha not in dec.read_text():
        with open(dec, "a") as fh:
            fh.write(linha + "\n")
print("OK:", BASE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OK: /content/drive/MyDrive/mixers-ethereum


## 1 — Classificação
Limiar do artigo: probabilidade > 0,5.

In [34]:
import joblib
modelo = joblib.load(PROC/"modelo_rf.joblib")
feat_teste = pd.read_parquet(PROC/"features_teste_20201029.parquet")
treino = pd.read_parquet(PROC/"features_treino.parquet")
MIN_TX = 10         # use o mesmo valor do notebook 03
LIMIAR = 0.5      # artigo: probabilidade > 0,5 = "possível mixer"

elegiveis = feat_teste[feat_teste.n_tx >= MIN_TX].copy()
elegiveis["prob_mixer"] = modelo.predict_proba(elegiveis[FEATURES])[:, 1]
elegiveis["sinalizado"] = elegiveis.prob_mixer > LIMIAR
print(f"Destinatários no dia: {len(feat_teste):,} | classificados (>= {MIN_TX} tx): {len(elegiveis):,}")

Destinatários no dia: 230,658 | classificados (>= 10 tx): 4,038


## 2 — Comparação com o artigo
Tabela principal para o slide de resultados.

In [35]:
sin = elegiveis[elegiveis.sinalizado]
tornado_ok = sin.tornado.notna().sum()
comparacao = pd.DataFrame([
    ("Endereços sinalizados",           63, len(sin)),
    ("Sinalizados que são Tornado",      3, tornado_ok),
    ("Falsos positivos",                60, len(sin) - tornado_ok),
    ("Precision",                  round(3/63, 3), round(tornado_ok / len(sin), 3) if len(sin) else np.nan),
    ("Recall (3 instâncias Tornado)", 1.0, round(tornado_ok / 3, 3)),
], columns=["métrica", "artigo", "reprodução"])
comparacao.to_csv(RES/"comparacao_artigo_20201029.csv", index=False)
print("Instâncias do Tornado:")
print(elegiveis[elegiveis.tornado.notna()][["tornado", "n_tx", "prob_mixer", "sinalizado"]].to_string(index=False))
comparacao

Instâncias do Tornado:
tornado  n_tx  prob_mixer  sinalizado
0.1 ETH    73        0.87        True
  1 ETH    55        0.87        True
 10 ETH   111        0.86        True


,métrica,artigo,reprodução
0,Endereços sinalizados,63.000,9.000
1,Sinalizados que são Tornado,3.000,3.000
2,Falsos positivos,60.000,6.000
3,Precision,0.048,0.333
4,Recall (3 instâncias Tornado),1.000,1.000


## 3 — Lista para análise dos falsos positivos
Gera a planilha com link do Etherscan de cada sinalizado. O grupo classifica cada um (DEX, NFT, pool de liquidez, GameFi...) e compara com as categorias do artigo.

In [36]:
fp = (sin.sort_values("prob_mixer", ascending=False)
         .assign(etherscan="https://etherscan.io/address/" + sin.sort_values("prob_mixer", ascending=False).recipient,
                categoria="", rotulo_etherscan="", observacao=""))
cols = ["recipient", "etherscan", "tornado", "prob_mixer", "n_tx"] + FEATURES + ["categoria", "rotulo_etherscan", "observacao"]
fp[cols].to_csv(RES/"sinalizados_20201029.csv", index=False)
print(f"{len(fp)} endereços em results/sinalizados_20201029.csv — importem no Google Sheets e preencham 'categoria'")
print("Sugestão de categorias (as do artigo): DEX, NFT, pool de liquidez, GameFi, exchange, outro")
fp[cols[:5]].head(15)

9 endereços em results/sinalizados_20201029.csv — importem no Google Sheets e preencham 'categoria'
Sugestão de categorias (as do artigo): DEX, NFT, pool de liquidez, GameFi, exchange, outro


,recipient,etherscan,tornado,prob_mixer,n_tx
86557,0x5e60fcca904753a31f9df7d7f6b7856b6ec27aa9,https://etherscan.io/address/0x5e60fcca904753a...,None,1.00,53
138954,0x98f1858c9740b9c5f2cd8abea67948873d38dc39,https://etherscan.io/address/0x98f1858c9740b9c...,None,0.98,10
200935,0xde7b3b2fe0e7b4925107615a5b199a4eb40d9ca9,https://etherscan.io/address/0xde7b3b2fe0e7b49...,None,0.91,52
17637,0x12d66f87a04a9e220743712ce6d9bb1b5616b8fc,https://etherscan.io/address/0x12d66f87a04a9e2...,0.1 ETH,0.87,73
66101,0x47ce0c6ed5b0ce3d3a51fdb1c52dc66a7c3c2936,https://etherscan.io/address/0x47ce0c6ed5b0ce3...,1 ETH,0.87,55
146385,0xa160cdab225685da1d56aa342ad8841c3b53f291,https://etherscan.io/address/0xa160cdab225685d...,None,0.87,29
131957,0x910cbd523d972eb0a6f4cae4618ad62622b39dbf,https://etherscan.io/address/0x910cbd523d972eb...,10 ETH,0.86,111
203271,0xe113d58a4cb4dbac8a0da608ed7232785c1f2c18,https://etherscan.io/address/0xe113d58a4cb4dba...,None,0.76,10
155862,0xabea9132b05a70803a4e85094fd0e1800777fbef,https://etherscan.io/address/0xabea9132b05a708...,None,0.59,25


## 4 — Sensibilidade
Como o número de sinalizados muda com o limiar de probabilidade e com o mínimo de transações por endereço. O artigo usa só limiar 0,5 e não informa o mínimo.

In [37]:
linhas = []
for mt in [MIN_TX, 10, 20]:
    for lim in [0.5, 0.7, 0.9]:
        e = elegiveis[elegiveis.n_tx >= mt]
        s = e[e.prob_mixer > lim]
        linhas.append({"MIN_TX": mt, "limiar": lim, "sinalizados": len(s),
                   "tornado_detectados": s.tornado.notna().sum(),
                       "falsos_positivos": len(s) - s.tornado.notna().sum()})
sens = pd.DataFrame(linhas)
sens.to_csv(RES/"sensibilidade_limiar_mintx.csv", index=False)
sens

,MIN_TX,limiar,sinalizados,tornado_detectados,falsos_positivos
0,10,0.5,9,3,6
1,10,0.7,8,3,5
2,10,0.9,3,0,3
3,10,0.5,9,3,6
4,10,0.7,8,3,5
5,10,0.9,3,0,3
6,20,0.5,7,3,4
7,20,0.7,6,3,3
8,20,0.9,2,0,2


## 5 — Variação entre sementes
O artigo mostra uma única execução. Aqui o modelo é treinado 5 vezes com sementes diferentes, para medir quanto o resultado varia.

In [38]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import resample

def reamostrar(X, y, seed):
    d = X.assign(_y=y.values); maj, mino = d[d._y == 0], d[d._y == 1]
    if len(mino) < len(maj):
        mino = resample(mino, replace=True, n_samples=len(maj), random_state=seed)
    d = pd.concat([maj, mino]); return d[FEATURES], d["_y"]

res = []
for seed in range(5):
    Xr, yr = reamostrar(treino[FEATURES], treino["label"], seed)
    m = RandomForestClassifier(n_estimators=100, max_depth=5, class_weight="balanced",
                               random_state=seed).fit(Xr, yr)
    p = m.predict_proba(elegiveis[FEATURES])[:, 1] > LIMIAR
    s = elegiveis[p]
    res.append({"seed": seed, "sinalizados": len(s), "tornado_detectados": s.tornado.notna().sum()})
sementes = pd.DataFrame(res).set_index("seed")
sementes = pd.concat([sementes, sementes.agg(["mean", "std"]).rename(index={"mean": "média", "std": "desvio"})])
sementes.to_csv(RES/"variacao_sementes.csv")
sementes.round(2)

,sinalizados,tornado_detectados
0,9.00,3.0
1,10.00,3.0
2,9.00,3.0
3,10.00,3.0
4,9.00,3.0
média,9.40,3.0
desvio,0.55,0.0


In [39]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import resample

def reamostrar(X, y, seed):
    d = X.assign(_y=y.values); maj, mino = d[d._y == 0], d[d._y == 1]
    if len(mino) < len(maj):
        mino = resample(mino, replace=True, n_samples=len(maj), random_state=seed)
    d = pd.concat([maj, mino]); return d.drop(columns="_y"), d["_y"]

def rodar(min_tx, feats, seed=42):
    tr = treino[treino.n_tx >= min_tx]
    Xr, yr = reamostrar(tr[feats], tr["label"], seed)
    m = RandomForestClassifier(n_estimators=100, max_depth=5, class_weight="balanced",
                               random_state=seed).fit(Xr, yr)
    e = feat_teste[feat_teste.n_tx >= min_tx]
    s = e[m.predict_proba(e[feats])[:, 1] > LIMIAR]
    return len(s), int(s.tornado.notna().sum())

sem_cc = [f for f in FEATURES if f != "call_count_mean"]
print("n_tx>=5 , 4 features      :", rodar(5, FEATURES))
print("n_tx>=10, 4 features      :", rodar(10, FEATURES))
print("n_tx>=10, sem call_count  :", rodar(10, sem_cc))
print("n_tx>=5 , sem call_count  :", rodar(5, sem_cc))
print("sinalizados com n_tx<10   :", (sin.n_tx < 10).sum(), "de", len(sin))
print("treino por n_tx>=10       :", treino[treino.n_tx >= 10].label.value_counts().to_dict())

n_tx>=5 , 4 features      : (14, 3)
n_tx>=10, 4 features      : (9, 3)
n_tx>=10, sem call_count  : (75, 3)
n_tx>=5 , sem call_count  : (410, 3)
sinalizados com n_tx<10   : 0 de 9
treino por n_tx>=10       : {0: 100, 1: 23}


---
## Resultado
Com as tabelas de `results/`, o grupo tem o material da seção de resultados e da discussão (falsos positivos, sensibilidade e estabilidade).